In [ ]:
import subprocess, os
os.makedirs("data", exist_ok=True)
subprocess.run(["python3", "dataset.py", "--out", "data/spam_dataset.csv"], check=True)
print("Dataset ready.")

In [ ]:
%%writefile train_and_export_model.py
import argparse
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", default="data/spam_dataset.csv")
    parser.add_argument("--out", default="data/model.joblib")
    args = parser.parse_args()

    df = pd.read_csv(args.data)
    X = df["text"]
    y = df["label"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    model = make_pipeline(TfidfVectorizer(), MultinomialNB())
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    print(f"accuracy={accuracy_score(y_test, preds):.4f}  f1={f1_score(y_test, preds, pos_label='spam'):.4f}")

    joblib.dump({"model": model, "feature_columns": ["text"]}, args.out)
    print(f"Saved model bundle to {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
subprocess.run(["python3", "train_and_export_model.py"], check=True)
print(os.path.getsize("data/model.joblib"), "bytes")

In [ ]:
%%writefile predictor_app.py
"""
predictor_app.py — AI Operations (AIOps), Module 3 Lecture 2b
FastAPI predictor implementing the KServe V1 inference protocol.
"""
import redis
import os, socket, time
import joblib
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

MODEL_PATH = os.environ.get("MODEL_PATH", "data/model.joblib")
POD_NAME = os.environ.get("POD_NAME", socket.gethostname())
NODE_NAME = os.environ.get("NODE_NAME", "unknown")

app = FastAPI(title="Email Classifier Predictor")
_bundle = None

REDIS_HOST = os.environ["REDIS_HOST"]
redis_client = redis.Redis(host=REDIS_HOST, port=6379, decode_responses=True)

@app.on_event("startup")
def load_model():
    global _bundle
    _bundle = joblib.load(MODEL_PATH)
    print(f"Loaded model ({len(_bundle['feature_columns'])} features) on pod={POD_NAME} node={NODE_NAME}")


class PredictRequest(BaseModel):
    instances: list[str] 


@app.get("/healthz")
def healthz():
    return {"status": "ok", "pod": POD_NAME, "node": NODE_NAME}

 

@app.post("/v1/models/{model_name}:predict")
def predict(model_name: str, request: PredictRequest):
    if _bundle is None:
        raise HTTPException(status_code=503, detail="Model not loaded yet")
    
    model = _bundle["model"]  
    
    t0 = time.time()
    predictions = []
    for test_case in request.instances:
        if redis_client.exists(test_case):
            prediction = redis_client.get(test_case)
        else:
            prediction = model.predict([test_case])[0]
            redis_client.set(test_case,prediction, ex = 1800)
        predictions.append(prediction)
    latency_ms = round((time.time() - t0) * 1000, 2)

    return {
        "predictions": predictions,
        "served_by_pod": POD_NAME,
        "served_by_node": NODE_NAME,
        "latency_ms": latency_ms,
    }

In [ ]:
import subprocess, time, requests, os

env = os.environ.copy()
env.update({"MODEL_PATH": "data/model.joblib", "POD_NAME": "local-test", "NODE_NAME": "local"})

proc = subprocess.Popen(
    ["uvicorn", "predictor_app:app", "--host", "0.0.0.0", "--port", "28080"],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

In [ ]:
health = requests.get("http://localhost:28080/healthz").json()
print("Health check:", health)

payload = {"instances": [[0.5] * 20]}
pred = requests.post("http://localhost:28080/v1/models/rf-classifier:predict", json=payload).json()
print("Prediction:", pred)